# CoalitionGNN — Koalisyon Örnekleri ve Hedef Gözlemler

Bu notebook üç şey üretir:

1. **Pozitif koalisyon örnekleri** — `coalitions_canon.yaml`'dan okunur, koalisyon-yıl panelleri olarak açılır
2. **Üç hedef gözlem** her koalisyon-yıl için:
   - Süre (kaç yıl yaşadı, sansürlü olabilir)
   - İçsel çatışma sayısı (MIDB'den koalisyon üyeleri arasında)
   - İçsel oylama uyumu (ideal point ortalama yakınlığı)
3. **Negatif örnekler** — sert (yakın ama gerçekleşmemiş) + rastgele

**Ön koşul:** `02_preprocessing.ipynb` çalıştırılmış, `data/canonical/` dolu.

**Çıktı:** `data/coalitions/` altında parquet dosyaları + train/val/test bölmesi.

## 0. Hazırlık

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
RAW_DIR = os.path.join(PROJECT_DIR, 'data/raw')
CANON_DIR = os.path.join(PROJECT_DIR, 'data/canonical')
COAL_DIR = os.path.join(PROJECT_DIR, 'data/coalitions')
os.makedirs(COAL_DIR, exist_ok=True)

# Manifest v3 ile uyumlu sınırlar
START_YEAR = 1946
END_YEAR = 2016

# Train/val/test bölmesi — manifestte 2000/2010 önerilmişti,
# 2016 sınırı yüzünden val penceresi daraldı
TRAIN_END = 1999
VAL_END = 2009
# TEST: 2010-2016 (7 yıl)

print(f'Kanon dizini: {CANON_DIR}')
print(f'Çıktı dizini: {COAL_DIR}')
print(f'Dönem: {START_YEAR}-{END_YEAR}')
print(f'Bölme: train={START_YEAR}-{TRAIN_END}, val={TRAIN_END+1}-{VAL_END}, test={VAL_END+1}-{END_YEAR}')

In [ ]:
!pip install -q pyyaml pandas pyarrow tqdm numpy

In [ ]:
import yaml
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

# Sonuç toplayıcı
stages = []

def log_stage(name, status, details, n=None):
    stages.append({'stage': name, 'status': status, 'details': details, 'n': n})
    icon = {'OK': '✅', 'WARN': '⚠️', 'FAIL': '❌'}.get(status, '?')
    suffix = f' [{n:,}]' if n is not None else ''
    print(f'{icon} {name}: {details}{suffix}')

## 1. Kanonu Yükle ve Doğrula

Kanon YAML dosyası `data/canonical/coalitions_canon.yaml` konumunda olmalı.
Eğer Drive'da yoksa, önce buraya yükle.

In [ ]:
canon_path = os.path.join(CANON_DIR, 'coalitions_canon.yaml')

if not os.path.exists(canon_path):
    raise FileNotFoundError(
        f'Kanon dosyası bulunamadı: {canon_path}\n'
        f'Önce coalitions_canon.yaml dosyasını şuraya yükle: {CANON_DIR}'
    )

with open(canon_path, 'r', encoding='utf-8') as f:
    canon = yaml.safe_load(f)

coalitions = canon['coalitions']
log_stage('1_canon_load', 'OK',
          f'{len(coalitions)} koalisyon yüklendi',
          n=len(coalitions))

# Tip dağılımı
from collections import Counter
type_dist = Counter(c.get('type', '?') for c in coalitions)
print(f'\nTip dağılımı: {dict(type_dist)}')

# Birkaç örnek
print('\nKanon örnekleri:')
for c in coalitions[:3]:
    print(f"  {c['id']:20s} | {c['name']:40s} | {c['start_year']}-{c['end_year']} | {len(c['members'])} üye")

In [ ]:
# Kanon doğrulaması: COW kodları country_master'da var mı?
country_master = pd.read_parquet(os.path.join(CANON_DIR, 'country_master.parquet'))
valid_cows = set(country_master['cow_code'].unique())

validation_issues = []
for c in coalitions:
    invalid = [m for m in c['members'] if m not in valid_cows]
    if invalid:
        validation_issues.append((c['id'], invalid))

if validation_issues:
    log_stage('1_canon_validate', 'WARN',
              f'{len(validation_issues)} koalisyonda geçersiz COW kodu var')
    for cid, codes in validation_issues[:10]:
        print(f'  {cid}: geçersiz kodlar = {codes}')
    if len(validation_issues) > 10:
        print(f'  ... ve {len(validation_issues)-10} daha')
else:
    log_stage('1_canon_validate', 'OK', 'tüm COW kodları geçerli')

## 2. Koalisyon-Yıl Panelini Aç

Her koalisyon, kapsadığı her yıl için bir satır olur. Bu, hedef gözlemlerin hesaplanacağı tablo.

In [ ]:
# Koalisyon-yıl paneli
panel_rows = []
for c in coalitions:
    start = c['start_year']
    # end_year null ise hâlâ aktif; END_YEAR'a kadar uzat (sansürlü)
    end = c['end_year'] if c['end_year'] is not None else END_YEAR
    censored = c['end_year'] is None  # sağ-sansürlü mü?
    
    for y in range(max(start, START_YEAR), min(end, END_YEAR) + 1):
        # O yıl gerçekten var olan üyeleri filtrele
        members_y = [m for m in c['members']
                     if ((country_master['cow_code'] == m) &
                         (country_master['year'] == y)).any()]
        if len(members_y) < 2:
            continue  # 2'den az üye → koalisyon değil
        
        panel_rows.append({
            'coalition_id': c['id'],
            'coalition_name': c['name'],
            'type': c.get('type', '?'),
            'region': c.get('region', '?'),
            'year': y,
            'start_year': start,
            'end_year': end,
            'censored': censored,
            'members': sorted(members_y),
            'n_members': len(members_y),
        })

panel = pd.DataFrame(panel_rows)
log_stage('2_panel_built', 'OK',
          f'koalisyon-yıl paneli',
          n=len(panel))

# İstatistik
print(f'\nYıl başına ortalama koalisyon sayısı: {panel.groupby("year").size().mean():.1f}')
print(f'Koalisyon başına ortalama yıl: {panel.groupby("coalition_id").size().mean():.1f}')
print(f'Sansürlü kayıt oranı: {panel["censored"].mean():.1%}')
print(f'\nÜye sayısı dağılımı:')
print(panel['n_members'].describe().round(1))

## 3. Hedef Gözlem 1 — Koalisyon Süresi

Her koalisyon-yıl satırı için 'şu ana kadar yaşadığı süre' ve 'sağ-sansürlü mü' işareti.
Survival analizinde standart format: (time, event), event=1 → koalisyon sonlandı, event=0 → hâlâ devam ediyor.

In [ ]:
# Her satır için: o yıl itibarıyla koalisyon kaç yıldır var
panel['age_years'] = panel['year'] - panel['start_year']
panel['total_duration'] = panel['end_year'] - panel['start_year'] + 1
# event = 1 → o yıl bitti (panel'in son yılı, sansürsüz)
# event = 0 → hâlâ devam ediyor
panel['is_terminal_year'] = (panel['year'] == panel['end_year'])
panel['event'] = ((panel['is_terminal_year']) & (~panel['censored'])).astype(int)

log_stage('3a_survival', 'OK',
          f"event=1 (sonlandı): {panel['event'].sum()}, sansürlü: {(~panel['event'].astype(bool) & panel['is_terminal_year']).sum()}")

print(panel[['coalition_id', 'year', 'age_years', 'total_duration', 'event', 'censored']].head(10))

## 4. Hedef Gözlem 2 — Koalisyon-İçi Çatışma

MIDB veritabanından, koalisyon üyeleri arasında o yıl kaç MID var.
Düşük sayı = stabil koalisyon, yüksek sayı = içsel gerilim.

In [ ]:
# edges_conflict zaten ülke çifti × yıl bazında MID sayımı tutuyor (log dönüştürülmüş)
edges_conflict = pd.read_parquet(os.path.join(CANON_DIR, 'edges_conflict.parquet'))

# Her koalisyon-yıl için içsel çift sayısı ve toplam ağırlık
def internal_conflict(members, year, conflict_df):
    members_set = set(members)
    sub = conflict_df[
        (conflict_df['year'] == year) &
        (conflict_df['cow_i'].isin(members_set)) &
        (conflict_df['cow_j'].isin(members_set))
    ]
    if len(sub) == 0:
        return 0.0, 0
    return float(sub['weight'].sum()), len(sub)

# Tüm panele uygula (yıllara göre indeksli arama hız kazandırır)
conflict_by_year = {y: g for y, g in edges_conflict.groupby('year')}

results = []
for _, row in tqdm(panel.iterrows(), total=len(panel), desc='İçsel çatışma'):
    y = row['year']
    members = row['members']
    cf = conflict_by_year.get(y, pd.DataFrame())
    if len(cf) == 0:
        results.append((0.0, 0))
        continue
    members_set = set(members)
    sub = cf[cf['cow_i'].isin(members_set) & cf['cow_j'].isin(members_set)]
    results.append((float(sub['weight'].sum()), len(sub)))

panel['internal_conflict_weight'] = [r[0] for r in results]
panel['internal_conflict_pairs'] = [r[1] for r in results]

log_stage('4_internal_conflict', 'OK',
          f"içsel çatışma çiftleri (toplam): {panel['internal_conflict_pairs'].sum()}")

# Üst birkaç çatışma içeren koalisyon-yıl
top_conflict = panel.nlargest(10, 'internal_conflict_weight')[
    ['coalition_id', 'year', 'n_members', 'internal_conflict_pairs', 'internal_conflict_weight']
]
print('\nEn fazla içsel çatışma yaşayan koalisyon-yıllar:')
print(top_conflict)

## 5. Hedef Gözlem 3 — Koalisyon-İçi Oylama Uyumu

BM oylama yakınlık kenarlarından, koalisyon üyeleri arasındaki ortalama yakınlık.
Yüksek değer = ideolojik uyum, düşük = parçalı koalisyon.

In [ ]:
edges_vote = pd.read_parquet(os.path.join(CANON_DIR, 'edges_vote.parquet'))
vote_by_year = {y: g for y, g in edges_vote.groupby('year')}

agreements = []
coverages = []

for _, row in tqdm(panel.iterrows(), total=len(panel), desc='İçsel oylama uyumu'):
    y = row['year']
    members = row['members']
    vy = vote_by_year.get(y, pd.DataFrame())
    
    if len(vy) == 0 or len(members) < 2:
        agreements.append(np.nan)
        coverages.append(0.0)
        continue
    
    members_set = set(members)
    sub = vy[vy['cow_i'].isin(members_set) & vy['cow_j'].isin(members_set)]
    
    n_pairs = len(members) * (len(members) - 1) // 2
    if len(sub) == 0:
        agreements.append(0.0)  # mesafe büyük (eşik dışı) → düşük yakınlık
        coverages.append(0.0)
    else:
        agreements.append(float(sub['weight'].mean()))
        coverages.append(len(sub) / n_pairs)

panel['internal_vote_agreement'] = agreements
panel['internal_vote_coverage'] = coverages

log_stage('5_vote_agreement', 'OK',
          f"ortalama uyum: {panel['internal_vote_agreement'].mean():.3f}, kapsama: {panel['internal_vote_coverage'].mean():.1%}")

# En yüksek uyum
top_agree = panel.nlargest(10, 'internal_vote_agreement')[
    ['coalition_id', 'year', 'n_members', 'internal_vote_agreement', 'internal_vote_coverage']
]
print('\nEn yüksek içsel oylama uyumu (yıl × koalisyon):')
print(top_agree)

## 6. Pozitif Örneklerin Kaydı

In [ ]:
# Pozitif örnekler — model bunlardan öğrenir
# Üye listesini saklamak için string formatına çevir (parquet listeleri parquet'ta sorun çıkarır)
panel_save = panel.copy()
panel_save['members_str'] = panel_save['members'].apply(lambda x: ','.join(map(str, x)))
panel_save = panel_save.drop(columns=['members'])

panel_save.to_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
log_stage('6_positives_saved', 'OK', f'positive_samples.parquet', n=len(panel_save))

## 7. Negatif Örnekleyiciler

İki tür:
- **Sert negatif:** Pozitif koalisyonun üye listesinde bir veya iki ülkeyi değiştir (örn. NATO + Rusya yerine).
- **Rastgele negatif:** Aynı büyüklükte rastgele ülke kümesi.

Negatif örnekler için hedef gözlem hesaplanmaz — onlar sadece **oyun-teorik kayıplara** (üst-katkılılık, sıralama) girdi olur.

In [ ]:
rng = np.random.default_rng(42)

def generate_hard_negatives(panel_row, n_swaps=2, n_per_pos=2):
    """Pozitif koalisyondan k üye çıkarıp k yeni ülke ekleyerek sert negatif üret."""
    members = panel_row['members']
    year = panel_row['year']
    
    # O yıl mevcut olan ülkeler
    available = set(country_master[country_master['year'] == year]['cow_code'])
    candidates_outside = list(available - set(members))
    
    if len(candidates_outside) < n_swaps or len(members) < n_swaps + 2:
        return []
    
    negs = []
    for _ in range(n_per_pos):
        out_idx = rng.choice(len(members), size=n_swaps, replace=False)
        in_idx = rng.choice(len(candidates_outside), size=n_swaps, replace=False)
        new_members = [m for i, m in enumerate(members) if i not in out_idx]
        new_members += [candidates_outside[i] for i in in_idx]
        negs.append(sorted(new_members))
    return negs

def generate_random_negatives(panel_row, n_per_pos=2):
    """Aynı büyüklükte tamamen rastgele ülke kümesi (pozitif üye listesiyle hiç örtüşmesin)."""
    members = panel_row['members']
    year = panel_row['year']
    available = list(set(country_master[country_master['year'] == year]['cow_code']) - set(members))
    
    if len(available) < len(members):
        return []
    
    negs = []
    for _ in range(n_per_pos):
        idx = rng.choice(len(available), size=len(members), replace=False)
        negs.append(sorted([available[i] for i in idx]))
    return negs

In [ ]:
# Tüm pozitif örnekler için sert ve rastgele negatif üret
negative_rows = []

for _, row in tqdm(panel.iterrows(), total=len(panel), desc='Negatif örnekler'):
    for hard_neg in generate_hard_negatives(row, n_swaps=2, n_per_pos=2):
        negative_rows.append({
            'source_coalition_id': row['coalition_id'],
            'year': row['year'],
            'members_str': ','.join(map(str, hard_neg)),
            'n_members': len(hard_neg),
            'neg_type': 'hard',
        })
    for rand_neg in generate_random_negatives(row, n_per_pos=1):
        negative_rows.append({
            'source_coalition_id': row['coalition_id'],
            'year': row['year'],
            'members_str': ','.join(map(str, rand_neg)),
            'n_members': len(rand_neg),
            'neg_type': 'random',
        })

negatives = pd.DataFrame(negative_rows)
negatives.to_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

log_stage('7_negatives', 'OK',
          f'hard: {(negatives["neg_type"]=="hard").sum()}, random: {(negatives["neg_type"]=="random").sum()}',
          n=len(negatives))

## 8. Train/Val/Test Bölmesi

Manifestte belirtildiği gibi zamansal bölme: 1946-1999 / 2000-2009 / 2010-2016.
Aynı koalisyon farklı yıllarda farklı setlerde olabilir — bu kasıtlı, çünkü 'koalisyonun zaman içindeki evrimi' modellenecek bir şey.

In [ ]:
def assign_split(year):
    if year <= TRAIN_END:
        return 'train'
    elif year <= VAL_END:
        return 'val'
    else:
        return 'test'

panel_save['split'] = panel_save['year'].apply(assign_split)
negatives['split'] = negatives['year'].apply(assign_split)

panel_save.to_parquet(os.path.join(COAL_DIR, 'positive_samples.parquet'))
negatives.to_parquet(os.path.join(COAL_DIR, 'negative_samples.parquet'))

# Bölme istatistikleri
print('Pozitif örnek bölmesi:')
print(panel_save['split'].value_counts().sort_index())
print('\nNegatif örnek bölmesi:')
print(negatives['split'].value_counts().sort_index())

log_stage('8_split', 'OK',
          f"train/val/test sınırları: {TRAIN_END}/{VAL_END}/{END_YEAR}")

## 9. Özet Rapor

In [ ]:
from datetime import datetime

report = pd.DataFrame(stages)
print('='*80)
print('KOALİSYON ÖRNEKLERİ — ÖZET RAPOR'.center(80))
print(f'{datetime.now().strftime("%Y-%m-%d %H:%M")}'.center(80))
print('='*80)

for status in ['OK', 'WARN', 'FAIL']:
    subset = report[report['status'] == status]
    if len(subset) == 0:
        continue
    icon = {'OK': '✅', 'WARN': '⚠️', 'FAIL': '❌'}[status]
    print(f'\n{icon} {status} ({len(subset)} aşama)')
    print('-'*80)
    for _, r in subset.iterrows():
        suffix = f" [{int(r['n']):,}]" if pd.notna(r['n']) else ''
        print(f"  {r['stage']:30s} {r['details']}{suffix}")

print('\n' + '='*80)
print('Üretilen dosyalar:')
for f in sorted(os.listdir(COAL_DIR)):
    fpath = os.path.join(COAL_DIR, f)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {f}: {size_kb:.1f} KB')
print('='*80)

# Detaylı rapor
report_path = os.path.join(PROJECT_DIR, f'data/coalition_samples_report_{datetime.now().strftime("%Y%m%d_%H%M")}.csv')
report.to_csv(report_path, index=False)

## Sonraki adım

Şu üç dosya hazır:
1. `data/coalitions/positive_samples.parquet` — koalisyon-yıl paneli + 3 hedef gözlem
2. `data/coalitions/negative_samples.parquet` — sert ve rastgele negatifler
3. Train/val/test bölmesi her iki tabloda da bir sütun olarak

**Veri tarafı tamamlandı.** Sonraki notebook (`04_model.ipynb`) artık model tarafı:
- Heterojen MPNN encoder
- DeepSets koalisyon okuyucu
- Üç gözlem başlığı (multi-task)
- Oyun-teorik kayıplar (çekirdek-hinge, üst-katkılılık)

**Şimdi bakılması gereken iki kalibrasyon noktası:**
- Kanonda 31 koalisyon var; bazıları az yıllı (1-2 yıl), bazıları çok uzun (50+). Süre dağılımı dengeli mi?
- İçsel oylama uyumu için kapsama oranı düşük olabilir — eşik mesafe 2.0 kenar tablosuna girmemiş çiftler oluyor. Eşik daha gevşek tutulup tekrar üretilebilir.